# INFO-6147-(01)-26W Deep Learning with Pytorch

## Project:

**Student Name:** Yun-Jiung Wang

**Student Number:** 1256222

**Date:** March 30th, 2026

**Description:**

This project is aim to allow user upload the photos of food and describe what the food tastes like. It will be helpful when traveling or when people wants to try international food, but not having a person to explain to you what that is.
Here comes this AI project, to assist people on observing new foods.

## Setup Env

In [ ]:
!pip install datasets gradio streamlit groq pyngrok

# dlownload Cloudflare tunnle tool
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

## Import Libs

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torchvision.datasets import Food101
from torch.utils.data import Dataset, Subset,DataLoader

from datasets import load_dataset

import seaborn as sns
from sklearn.metrics import confusion_matrix, f1_score

import io
from tqdm import tqdm
import streamlit as st
from groq import Groq

from google.colab import userdata,runtime,drive # Store the API via Colab Secret
import zipfile
import os

## Check Device

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(DEVICE)

## Setup Hyperparameters

In [ ]:
EPOCHS=50
BATCH_SIZE = 256
LEARNING_RATE=0.001
NUM_WORKERS= 4

DATA_PATH="/data"

## Load Dataset (Colab)

In [ ]:
# Transformer

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load dataset (train and test)
train_dataset = datasets.Food101(root=DATA_PATH, split='train', download=True, transform=train_transform)

test_dataset = datasets.Food101(root=DATA_PATH, split='test', download=True, transform=test_transform)

## Define Food Types for subset

In [ ]:
select_food_list = [
     # --- Asia ---
    'bibimbap', 'gyoza', 'sashimi', 'pad_thai', 'pho',
    'miso_soup', 'edamame', 'spring_rolls', 'sushi', 'dumplings',
    'hummus', 'falafel', 'baklava', 'chicken_curry', 'fried_rice',

    # --- Lattino ---
    'tacos', 'enchiladas', 'guacamole', 'ceviche', 'nachos',

    # --- Classic ---
    'pizza', 'hamburger', 'hot_dog', 'steak', 'french_fries',
    'grilled_salmon', 'spaghetti_bolognese', 'lasagna', 'club_sandwich',

    # --- Hot in Social Media ---
    'tiramisu', 'cheesecake', 'macarons', 'donuts', 'waffles',
    'pancakes', 'ice_cream', 'apple_pie', 'strawberry_shortcake',

    # --- seafoods ---
    'mussels', 'oysters'
]

class_to_idx = {cls_name: i for i, cls_name in enumerate(select_food_list)}
idx_to_class = {i: cls_name for cls_name, i in class_to_idx.items()}

class FilteredFood101(Dataset):
    def __init__(self, dataset, selected_classes, transform=None):
        self.dataset = dataset
        self.transform = transform

        self.selected_classes = selected_classes
        self.class_to_idx = {cls: i for i, cls in enumerate(selected_classes)}

        self.filtered_indices = []

        for i in range(len(dataset)):
            label = dataset._labels[i]
            class_name = dataset.classes[label]

            if class_name in self.selected_classes:
                self.filtered_indices.append(i)

    def __len__(self):
        return len(self.filtered_indices)

    def __getitem__(self, idx):
        real_idx = self.filtered_indices[idx]
        img, label = self.dataset[real_idx]

        class_name = self.dataset.classes[label]
        new_label = self.class_to_idx[class_name]  # ⭐ 重建label

        if self.transform:
            img = self.transform(img)

        return img, new_label

## Create Subset

In [ ]:
filtered_train = FilteredFood101(train_dataset, select_food_list)
filtered_test = FilteredFood101(test_dataset, select_food_list)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

test_loader = DataLoader(test_dataset,
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(len(filtered_train), len(filtered_test))

train_loader = DataLoader(filtered_train, batch_size=32, shuffle=True)
test_loader = DataLoader(filtered_test, batch_size=32, shuffle=False)

labels = [label for _, label in filtered_train]
print(min(labels), max(labels))

num_classes = max(labels)+1
print(f"num of classes: {num_classes}, max lavel:{max(labels)}")

## Build the model

In [ ]:
model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
print(f"num_classes:{num_classes}")
# model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# model.fc = nn.Sequential(
#     nn.Linear(model.fc.in_features, 512),
#     nn.BatchNorm1d(512),
#     nn.ReLU(),                # activate param
#     nn.Dropout(0.3),          # drop off 30%
#     nn.Linear(512, num_classes) # output classes
# )

model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 1024), # larger to 1024
    nn.BatchNorm1d(1024),
    nn.ReLU(),
    nn.Dropout(0.4),           # prevernt overfitting
    nn.Linear(1024, 512),      # add layer
    nn.BatchNorm1d(512),
    nn.ReLU(),
    nn.Linear(512, num_classes)         #final classes
)

model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE,weight_decay=1e-4)

## Define the Model

In [ ]:
class FoodClassifier:
    def __init__(self, model, train_loader, test_loader, criterion, optimizer, device, patience=5):
        self.model = model
        self.train_loader = train_loader
        self.test_loader = test_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.device = device
        self.patience = patience
        self.counter = 0
        self.best_loss = float('inf')
        self.history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

    def train_epoch(self):
        self.model.train()
        running_loss, correct, total = 0.0, 0, 0
        # Advoid process lines
        pbar = tqdm(self.train_loader, desc="[Train]", leave=False)
        for images, labels in pbar:
            images, labels = images.to(self.device), labels.to(self.device)
            self.optimizer.zero_grad()
            outputs = self.model(images)
            loss = self.criterion(outputs, labels)
            loss.backward()
            self.optimizer.step()

            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        return running_loss / len(self.train_loader), 100. * correct / total

    def validate(self):
        self.model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in self.test_loader:
                images, labels = images.to(self.device), labels.to(self.device)
                outputs = self.model(images)
                loss = self.criterion(outputs, labels)

                running_loss += loss.item()
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()
        return running_loss / len(self.test_loader), 100. * correct / total

    def run_training(self, num_epochs, save_path):
        print(f"Training started (ResNet-18 / 40 Classes)")
        print(f"Save destination: {save_path}")

        for epoch in range(num_epochs):
            t_loss, t_acc = self.train_epoch()
            v_loss, v_acc = self.validate()

            self.history['train_loss'].append(t_loss)
            self.history['train_acc'].append(t_acc)
            self.history['test_loss'].append(v_loss)
            self.history['test_acc'].append(v_acc)

            print(f"Epoch {epoch+1}/{num_epochs} | Train Acc: {t_acc:.2f}% | Test Acc: {v_acc:.2f}% | Test Loss: {v_loss:.4f}")

            # --- Early Stopping Logic ---
            if v_loss < self.best_loss:
                self.best_loss = v_loss
                self.counter = 0
                torch.save(self.model.state_dict(), save_path)
                print(f"🌟 New best model saved!")
            else:
                self.counter += 1
                print(f"⚠️ No improvement ({self.counter}/{self.patience})")
                if self.counter >= self.patience:
                    print(f"🛑 Early stopping at epoch {epoch+1}.")
                    break
        print(f"✅ Training complete.")

## Visualize Loss and Accuracy Curve

In [ ]:
def plot_training_results(history):
    epochs = range(1, len(history['train_loss']) + 1)
    plt.figure(figsize=(14, 5))

    # Loss
    plt.subplot(1, 2, 1)
    plt.plot(epochs, history['train_loss'], 'b-o', label='Training Loss')
    plt.plot(epochs, history['test_loss'], 'r--s', label='Validation Loss')
    plt.title('Model Convergence (Loss)')
    plt.xlabel('Epochs')
    plt.ylabel('Loss Value')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)

    # Accuracy
    plt.subplot(1, 2, 2)
    plt.plot(epochs, history['train_acc'], 'g-o', label='Training Acc')
    plt.plot(epochs, history['test_acc'], 'm--s', label='Validation Acc')
    plt.title('Model Performance (Accuracy)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.grid(True, linestyle=':', alpha=0.6)

    plt.tight_layout()
    plt.show()

## Define result visualization function

In [ ]:
def visualize_results(model, dataset, num_images=6):
    model.eval()
    fig = plt.figure(figsize=(15, 10))
    class_names = dataset.classes

    # select randomly
    indices = np.random.choice(len(dataset), num_images, replace=False)

    with torch.no_grad():
        for i, idx in enumerate(indices):
            image, label = dataset[idx]
            input_tensor = image.unsqueeze(0).to(DEVICE)

            output = model(input_tensor)
            _, pred = torch.max(output, 1)

            # transfer into showable picture
            img_display = image.permute(1, 2, 0).numpy()
            mean = np.array([0.485, 0.456, 0.406])
            std = np.array([0.229, 0.224, 0.225])
            img_display = std * img_display + mean
            img_display = np.clip(img_display, 0, 1)

            ax = plt.subplot(2, 3, i + 1)
            color = 'green' if pred.item() == label else 'red'
            ax.set_title(f"Pred: {class_names[pred.item()]}\nActual: {class_names[label]}", color=color)
            plt.imshow(img_display)
            plt.axis('off')
    plt.show()

## Confusion Matrix function

In [ ]:
def plot_confusion_matrix(model, dataloader, class_names, device):
    """
    model: trained model
    dataloader: test/validate DataLoader
    class_names: class name list
    device: 'cuda' (A100/L4) or 'cpu'
    """
    model.eval()
    all_preds = []
    all_labels = []

    print("Processing...")
    with torch.no_grad():
        for images, labels in dataloader:
            images = images.to(device)
            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Calculate Matrix
    cm = confusion_matrix(all_labels, all_preds)

    # grapthing
    plt.figure(figsize=(20, 15))
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)

    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.title('Food-40 Confusion Matrix', fontsize=16)
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.show()


## Validate Labels

In [ ]:
images, labels = next(iter(train_loader))
max_label = labels.max().item()
min_label = labels.min().item()

print(f"Range of label:{min_label} to {max_label}")
print(f"Output layers: {num_classes}")

## Start training

In [ ]:
trainer = FoodClassifier(
    model=model,
    train_loader=train_loader,
    test_loader=test_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=DEVICE
)

In [ ]:
trainer.run_training(num_epochs=EPOCHS,save_path="/data/food_best.pth")

## Show Evaluation

In [ ]:
from datasets.iterable_dataset import pl
# graphes
plot_training_results(trainer.history)
# Predict Label and Actual Label
visualize_results(model, test_dataset)
# Confustion Matrix
plot_confusion_matrix(model, test_loader, test_dataset.classes, DEVICE)

## Create UI

In [ ]:
%%writefile app.py
import streamlit as st
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
from groq import Groq
import os

# --- 1. Model Loading Configuration ---
# Use @st.cache_resource to avoid reloading the model on every interaction
@st.cache_resource
def load_food_model(model_path, num_classes=40):
    # Reconstruct the exact same ResNet-18 architecture used during training

    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 512),
        nn.BatchNorm1d(512),
        nn.ReLU(),                # activate param
        nn.Dropout(0.3),          # drop off 30%
        nn.Linear(512, num_classes) # output classes
    )


    # Load weights to CPU or GPU automatically
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if os.path.exists(model_path):
        state_dict = torch.load(model_path, map_location=device)
        model.load_state_dict(state_dict)
        print(f"✅ Model loaded successfully from {model_path}")
    else:
        print(f"⚠️ Model file not found at {model_path}. Using uninitialized weights.")

    model.to(device)
    model.eval()
    return model, device

# --- 2. Image Prediction Logic ---
def predict_dish(image, model, device, class_names):
    # Transformation must match your Training/Validation transforms
    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    img_tensor = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        outputs = model(img_tensor)
        _, predicted = outputs.max(1)
        index = predicted.item()
    return class_names[index]

# --- 3. Llama AI Guide Function ---
def ask_llama_chef(food_name, api_key):
    if not api_key:
        return "❌ Please enter your Groq API Key in the sidebar!"
    try:
        client = Groq(api_key=api_key)
        # Professional prompt for a travel guide persona
        prompt = f"""
        You are a witty and expert travel guide for a Canadian tourist.
        The AI vision model has identified this dish as '{food_name}'.
        Please provide: 1. Taste & Texture, 2. A fun Travel Trivia fact, 3. Practical advice for travelers.
        Tone: Friendly and humorous. Length: Under 150 words.
        """
        completion = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"Error connecting to Llama: {str(e)}"

# --- 4. Streamlit UI Interface ---
st.set_page_config(page_title="Food Guide", page_icon="🍔", layout="centered")

# Sidebar for API Key and Settings
with st.sidebar:
    st.title("🛠️ Settings")
    user_api_key = st.text_input("Groq API Key", type="password")
    st.markdown("---")
    st.info("The model recognizes **40 types** of dishes.")

st.title("Food Guide")
st.write("Upload a photo of your meal, and we will identify it and tell you its story!")

# --- CONFIGURATION: Update these after your training finishes ---
# Path to the .pth file you are saving tonight
MODEL_FILE = "/data/food_best.pth"
# IMPORTANT: This list must match the EXACT order of your label_map (0 to 38)
select_food_list = [
     # --- Asia ---
    'bibimbap', 'gyoza', 'sashimi', 'pad_thai', 'pho',
    'miso_soup', 'edamame', 'spring_rolls', 'sushi', 'dumplings',
    'hummus', 'falafel', 'baklava', 'chicken_curry', 'fried_rice',

    # --- Lattino ---
    'tacos', 'enchiladas', 'guacamole', 'ceviche', 'nachos',

    # --- Classic ---
    'pizza', 'hamburger', 'hot_dog', 'steak', 'french_fries',
    'grilled_salmon', 'spaghetti_bolognese', 'lasagna', 'club_sandwich',

    # --- Hot in Social Media ---
    'tiramisu', 'cheesecake', 'macarons', 'donuts', 'waffles',
    'pancakes', 'ice_cream', 'apple_pie', 'strawberry_shortcake',

    # --- seafoods ---
    'mussels', 'oysters'
]

num_claeeses = len(select_food_list)

# --- File Uploader ---
uploaded_file = st.file_uploader("📸 Choose a food image...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    # Display the uploaded image
    image = Image.open(uploaded_file).convert('RGB')
    st.image(image, caption='Uploaded Image', use_container_width=True)

    if st.button("Identify & Get Guide"):
        with st.spinner("Analyzing your dish..."):
            # 1. Load the trained ResNet-18 model
            model, device = load_food_model(MODEL_FILE, num_classes=40)

            # 2. Run Inference
            predicted_label = predict_dish(image, model, device, select_food_list)
            st.success(f"Prediction: **{predicted_label.replace('_', ' ').title()}**")

            # 3. Generate Guide via Llama
            guide_result = ask_llama_chef(predicted_label, user_api_key)
            st.chat_message("assistant", avatar="👨‍🍳").write(guide_result)

## Activate Tunnel

In [ ]:
import subprocess
import time
import socket

# --- 1. Check if Streamlit is already running on port 8501 ---
def is_port_open(port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(('localhost', port)) == 0

# --- 2. Start Streamlit if it's not already running ---
if not is_port_open(8501):
    print("🚀 Starting Streamlit in the background...")
    subprocess.Popen(["streamlit", "run", "app.py", "--server.port", "8501", "--server.address", "0.0.0.0"],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    time.sleep(5) # Give it time to boot
else:
    print("✅ Streamlit is already running.")

# --- 3. Start Cloudflare Tunnel and FORCE print logs ---
print("🌐 Opening Cloudflare Tunnel... (Look for the '.trycloudflare.com' link below)")
print("-" * 50)

# We use stdbuf to disable buffering so the URL appears immediately
p = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://localhost:8501"],
                     stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)

# This loop will print EVERYTHING Cloudflare says until the URL appears
for line in p.stdout:
    print(f"DEBUG: {line.strip()}") # This helps us see if there's an error
    if "trycloudflare.com" in line:
        url = line.strip().split(" ")[-1]
        print("\n" + "★" * 50)
        print(f"🔥 SUCCESS! YOUR APP IS LIVE AT:")
        print(f"👉 {url}")
        print("★" * 50)
        # We don't break, so the tunnel stays active in this cell

# New Section

In [ ]:
print(filtered_train.dataset.classes)
print(len(filtered_train.dataset.classes))
print(len(num_class))